# Feature Engineering

In this notebook, we convert cleaned transactional and master data into
model-ready features that capture demand patterns, seasonality,
price effects, promotions, and supplier risk.

Each feature is created with a clear business purpose.

In [23]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

import sys
sys.path.append('../')

DATA_PATH_PROCESSED = "../data/processed"

In [ ]:
# Load processed data
from src.utils import load_processed_data

sales_df, inventory_df, products_df, suppliers_df = load_processed_data()

print("Processed data loaded successfully")

Processed data loaded successfully


In [5]:
# Convert date columns to datetime
sales_df["date"] = pd.to_datetime(sales_df["date"])

## Dataset Preparation

We merge sales, product, and supplier data to create a single
analysis-ready dataset.

In [6]:
df = (
    sales_df
    .merge(products_df, on="sku_id", how="left")
    .merge(suppliers_df, on="supplier_id", how="left")
)

df.head()

,date,sku_id,units_sold,selling_price,gross_revenue,promo_flag,discount_pct,month,week_of_year,day_of_week,is_weekend,is_festival_month,is_payday_period,season_tag,holiday_flag,weather_index,campaign_intensity,platform_traffic_source,traffic_index,competitor_price_index,competitor_stockout_flag,bundle_offer_flag,stock_visibility_score,rating_score,review_volume,product_visibility_rank,sku_name,category,sub_category,mrp,cost_price,supplier_id,supplier_name,avg_lead_time,lead_time_variability
0,2024-01-01,SKU0001,28,1028.54,28799.12,0,0.0072,1,1,0,0,0,1,winter,0,1.033050,1,affiliate,1.028220,1.039897,0,0,0.332970,4.99,220,14,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2
1,2024-01-02,SKU0001,27,942.76,25454.52,0,0.0900,1,1,1,0,0,1,winter,0,1.075864,0,organic,1.000124,1.034213,0,0,0.332574,4.99,220,26,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2
2,2024-01-03,SKU0001,27,1023.78,27642.06,0,0.0118,1,1,2,0,0,1,winter,0,0.776739,0,paid,1.042157,1.019800,0,0,0.369938,4.99,220,50,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2
3,2024-01-04,SKU0001,29,940.90,27286.10,0,0.0918,1,1,3,0,0,1,winter,0,1.029230,0,paid,0.998036,1.004657,0,0,0.445269,4.99,220,42,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2
4,2024-01-05,SKU0001,37,812.43,30059.91,1,0.2158,1,1,4,0,0,1,winter,0,1.093677,1,organic,1.047220,0.956571,0,0,0.414885,4.99,220,20,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2


## Calendar-Based Features

These features help capture regular purchasing patterns such as
month-start effects, festive periods, and quarterly cycles.

In [7]:
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

# Month start indicator (salary effect proxy)
df["is_month_start"] = df["date"].dt.day <= 5

## Lagged Demand Features

Lag features allow models to learn from recent demand history.
Weekly lags are used since inventory planning is done weekly.

In [10]:
df = df.sort_values(["sku_id", "date"])

for lag in [1, 2, 4]:
    df[f"lag_{lag}_week_sales"] = (
        df.groupby("sku_id")["units_sold"].shift(lag)
    )

## Rolling Demand Statistics

Rolling statistics capture local demand trends and volatility,
which are critical for safety stock calculations.

In [11]:
df["rolling_4w_mean"] = (
    df.groupby("sku_id")["units_sold"]
    .rolling(window=4)
    .mean()
    .reset_index(level=0, drop=True)
)

df["rolling_4w_std"] = (
    df.groupby("sku_id")["units_sold"]
    .rolling(window=4)
    .std()
    .reset_index(level=0, drop=True)
)

## Price & Promotion Features

Price changes and discounts strongly influence demand,
especially during campaigns and festive periods.

In [12]:
# Price gap (absolute discount)
df["price_gap"] = df["mrp"] - df["selling_price"]

# Discount percentage
df["discount_pct"] = df["price_gap"] / df["mrp"]

# Promotion flag (assuming discount > 0 implies promotion)
df["is_promo"] = df["discount_pct"] > 0

## Seasonality Indicators

Some categories experience strong seasonal demand.
We encode simple seasonality signals.

In [13]:
# Example seasonal mapping (adjust if needed)
winter_months = [11, 12, 1]
summer_months = [4, 5, 6]

df["is_winter_season"] = df["month"].isin(winter_months)
df["is_summer_season"] = df["month"].isin(summer_months)

## Supplier Lead Time & Risk Features

Lead time variability directly impacts reorder point and safety stock.

In [15]:
df.head()

,date,sku_id,units_sold,selling_price,gross_revenue,promo_flag,discount_pct,month,week_of_year,day_of_week,is_weekend,is_festival_month,is_payday_period,season_tag,holiday_flag,weather_index,campaign_intensity,platform_traffic_source,traffic_index,competitor_price_index,competitor_stockout_flag,bundle_offer_flag,stock_visibility_score,rating_score,review_volume,product_visibility_rank,sku_name,category,sub_category,mrp,cost_price,supplier_id,supplier_name,avg_lead_time,lead_time_variability,week,quarter,year,is_month_start,lag_1_week_sales,lag_2_week_sales,lag_4_week_sales,rolling_4w_mean,rolling_4w_std,price_gap,is_promo,is_winter_season,is_summer_season
0,2024-01-01,SKU0001,28,1028.54,28799.12,0,0.007201,1,1,0,0,0,1,winter,0,1.033050,1,affiliate,1.028220,1.039897,0,0,0.332970,4.99,220,14,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,NaN,NaN,NaN,NaN,NaN,7.46,True,True,False
1,2024-01-02,SKU0001,27,942.76,25454.52,0,0.090000,1,1,1,0,0,1,winter,0,1.075864,0,organic,1.000124,1.034213,0,0,0.332574,4.99,220,26,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,28.0,NaN,NaN,NaN,NaN,93.24,True,True,False
2,2024-01-03,SKU0001,27,1023.78,27642.06,0,0.011795,1,1,2,0,0,1,winter,0,0.776739,0,paid,1.042157,1.019800,0,0,0.369938,4.99,220,50,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,27.0,28.0,NaN,NaN,NaN,12.22,True,True,False
3,2024-01-04,SKU0001,29,940.90,27286.10,0,0.091795,1,1,3,0,0,1,winter,0,1.029230,0,paid,0.998036,1.004657,0,0,0.445269,4.99,220,42,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,27.0,27.0,NaN,27.75,0.957427,95.10,True,True,False
4,2024-01-05,SKU0001,37,812.43,30059.91,1,0.215801,1,1,4,0,0,1,winter,0,1.093677,1,organic,1.047220,0.956571,0,0,0.414885,4.99,220,20,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,29.0,27.0,28.0,30.00,4.760952,223.57,True,True,False


In [16]:
# Lead time is already present; create variability proxy
df["lead_time_days"] = df["avg_lead_time"]

# Supplier risk bucket
df["lead_time_risk"] = pd.cut(
    df["lead_time_days"],
    bins=[0, 5, 10, 20, np.inf],
    labels=["low", "medium", "high", "very_high"]
)

## Demand Volatility Feature

Volatility is measured using the coefficient of variation (CV).
High CV indicates unpredictable demand.

In [17]:
df["demand_cv"] = df["rolling_4w_std"] / df["rolling_4w_mean"]

## Final Feature Review

Lag and rolling features naturally introduce missing values.
These rows are removed for modeling purposes.

In [18]:
feature_df = df.dropna().reset_index(drop=True)

feature_df.shape

(14480, 51)

## Save Feature-Engineered Dataset

This dataset will be used for SKU segmentation and forecasting.

In [25]:
feature_df.to_csv(f"{DATA_PATH_PROCESSED}/feature_engineered_data.csv", index=False)

print("Feature engineering completed and saved successfully")

Feature engineering completed and saved successfully
